# Running Multiple Circuits on Hardware

This tutorial shows how a quantum circuit can be run on actual quantum hardware.

## Import all Necessary Libraries

In [1]:
from NoisyCircuits.QuantumCircuit import QuantumCircuit as nqc
from NoisyCircuits.utils.CreateNoiseModel import GetNoiseModel
from NoisyCircuits.RunOnHardware import RunOnHardware
import os
import numpy as np
import json

2026-07-09 17:24:05,812	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## Define all Necessary Parameters

In [ ]:
api_json = json.load(open(os.path.join(os.path.expanduser("~"), "ibm_api.json"), "r")) 
token = api_json["apikey"] # Replace with your Token
service_crn = api_json["service-crn"] # Replace with your Service CRN
backend_name = "ibm_fez"
shots = 2048
verbose = False
num_qubits = 2
num_trajectories = 500
qpu = "heron" # Only possible option for IBM backends at the moment
num_cores = 10
sim_backend = "custom" # Choose between "custom", "pennylane", "qulacs" and "qiskit"
use_fractional = True # Flag to determine whether to use fractional gates or not

## Get the Noise Model

In [3]:
noise_model = GetNoiseModel(backend_name, token, service_crn).get_noise_model()

Imputed T2 = T1 for qubit(s): [72]


## Build all Quantum Circuits

In [ ]:
circuit = nqc(
    num_qubits = num_qubits,
    noise_model = noise_model,
    backend_qpu_type = qpu,
    use_fractional = use_fractional,
    sim_backend = sim_backend,
    threshold = 1e-6,
    verbose = verbose
)

# List to store all probabilities from Pure State Simulation
probs_pure_list = []
# List to store all probabilities from MCWF Simulations
probs_sim_list = []

In [5]:
hardware_runner = RunOnHardware(
    token=token,
    backend=backend_name,
    shots=shots
)

qiskit_runtime_service._discover_account:WARNING:2026-07-09 17:24:39,408: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-07-09 17:24:41,784: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: Open_Sys. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-07-09 17:24:41,784: Loading instance: Open_Sys, plan: open


## Circuit 1: SWAP States

In [6]:
circuit.refresh()
circuit.RY(1.4, 0)
circuit.RY(2.5, 1)
circuit.SWAP(0, 1)
circuit.draw_circuit(style="text")

     ┌───┐┌────┐┌──────────┐┌────┐┌────┐   ┌────┐   ┌────┐   ┌───────┐┌───┐
q_0: ┤ X ├┤ √X ├┤ Rz(-1.4) ├┤ √X ├┤ √X ├─■─┤ √X ├─■─┤ √X ├─■─┤ Rx(π) ├┤ X ├
     ├───┤├────┤├──────────┤├────┤├────┤ │ ├────┤ │ ├────┤ │ └───────┘└───┘
q_1: ┤ X ├┤ √X ├┤ Rz(-2.5) ├┤ √X ├┤ √X ├─■─┤ √X ├─■─┤ √X ├─■───────────────
     └───┘└────┘└──────────┘└────┘└────┘   └────┘   └────┘                 


In [7]:
hardware_runner.create_circuits(
    circuit=circuit,
    measure_qubits=list(range(num_qubits))
)

In [8]:
probs_pure_list.append(circuit.run_pure_state(list(range(num_qubits)), num_cores=1))
probs_sim_list.append(circuit.execute(list(range(num_qubits)), num_trajectories=num_trajectories, num_cores=num_cores))

## Circuit 2: Parameterized Circuit

Set up a parameterized circuit with different set of parameters

In [9]:
parameter_set = [np.random.uniform(-2*np.pi, 2*np.pi, size=8) for _ in range(2)]

In [10]:
for circuit_num, params in enumerate(parameter_set):
    circuit.refresh()
    circuit.H(0)
    circuit.H(1)
    for i in range(len(params)//2):
        for j in range(num_qubits):
            circuit.RY(params[2*i+j], j)
        circuit.CZ(0, 1)
    circuit.draw_circuit(style="text")
    probs_pure_list.append(circuit.run_pure_state(list(range(num_qubits)), num_cores=1))
    probs_sim_list.append(circuit.execute(list(range(num_qubits)), num_trajectories=num_trajectories, num_cores=num_cores))
    hardware_runner.create_circuits(circuit=circuit, measure_qubits=list(range(num_qubits)))

     ┌────┐┌─────────┐┌────┐┌───┐┌────┐┌─────────────┐┌────┐   ┌───┐┌────┐»
q_0: ┤ √X ├┤ Rz(π/2) ├┤ √X ├┤ X ├┤ √X ├┤ Rz(-5.9178) ├┤ √X ├─■─┤ X ├┤ √X ├»
     ├────┤├─────────┤├────┤├───┤├────┤├─────────────┤├────┤ │ ├───┤├────┤»
q_1: ┤ √X ├┤ Rz(π/2) ├┤ √X ├┤ X ├┤ √X ├┤ Rz(-1.1514) ├┤ √X ├─■─┤ X ├┤ √X ├»
     └────┘└─────────┘└────┘└───┘└────┘└─────────────┘└────┘   └───┘└────┘»
«     ┌────────────┐┌────┐   ┌───┐┌────┐ ┌────────────┐┌────┐   ┌───┐┌────┐»
«q_0: ┤ Rz(2.0056) ├┤ √X ├─■─┤ X ├┤ √X ├─┤ Rz(5.6215) ├┤ √X ├─■─┤ X ├┤ √X ├»
«     ├────────────┤├────┤ │ ├───┤├────┤┌┴────────────┤├────┤ │ ├───┤├────┤»
«q_1: ┤ Rz(1.3509) ├┤ √X ├─■─┤ X ├┤ √X ├┤ Rz(-3.9539) ├┤ √X ├─■─┤ X ├┤ √X ├»
«     └────────────┘└────┘   └───┘└────┘└─────────────┘└────┘   └───┘└────┘»
«     ┌─────────────┐┌────┐   
«q_0: ┤ Rz(-1.0377) ├┤ √X ├─■─
«     ├─────────────┤├────┤ │ 
«q_1: ┤ Rz(-3.4424) ├┤ √X ├─■─
«     └─────────────┘└────┘   
     ┌────┐┌─────────┐┌────┐┌───┐┌────┐┌─────────────┐┌────┐   ┌───┐┌────┐»
q_0:

## Setup the Circuits for Submission

In [11]:
hardware_runner.setup_circuits()

qiskit_runtime_service.backends:WARNING:2026-07-09 17:24:51,506: Using instance: Open_Sys, plan: open
qiskit_runtime_service.backends:WARNING:2026-07-09 17:24:53,092: Using instance: Open_Sys, plan: open
qiskit_runtime_service.backends:WARNING:2026-07-09 17:24:54,129: Using instance: Open_Sys, plan: open


## Submit the Circuits

In [12]:
job_id = hardware_runner.run()

qiskit_runtime_service.backends:WARNING:2026-07-09 17:24:54,836: Using instance: Open_Sys, plan: open


Job ID: d97rqj0tcv6s73dlbhpg


## Query Job Status

In [14]:
hardware_runner.status()

'DONE'

## Job Cancellation

In the event the job hasn't already been completed, they can be cancelled.

In [ ]:
hardware_runner.cancel()

## Get Results

In [15]:
probs_hardware_list = hardware_runner.get_results()

## Compare Results

To compare the results we use the Battacharyya Coefficient defined for discrete probability distributions by:
\begin{equation}
BC(p, q) = \sum_{i=1}^N \sqrt{p_i q_i}
\end{equation}

In [16]:
def battacharyya_coefficient(p, q):
    return np.sum(np.sqrt(p * q))

In [17]:
for i in range(len(probs_pure_list)):
    print("Fidelity between Pure State Simulation and MCWF Simulation Results:\t", battacharyya_coefficient(probs_pure_list[i], probs_sim_list[i]))
    print("Fidelity between Pure State Simulation and Hardware Results:\t", battacharyya_coefficient(probs_pure_list[i], probs_hardware_list[i]))
    print("Fidelity between MCWF Simulation and Hardware Results:\t", battacharyya_coefficient(probs_sim_list[i], probs_hardware_list[i]))
    print("----------------------------------------------------------------------------------------------")

Fidelity between Pure State Simulation and MCWF Simulation Results:	 0.9963147833708492
Fidelity between Pure State Simulation and Hardware Results:	 0.9928456785170617
Fidelity between MCWF Simulation and Hardware Results:	 0.989754355432177
----------------------------------------------------------------------------------------------
Fidelity between Pure State Simulation and MCWF Simulation Results:	 0.9793725786176825
Fidelity between Pure State Simulation and Hardware Results:	 0.9693554510805642
Fidelity between MCWF Simulation and Hardware Results:	 0.9970000537346755
----------------------------------------------------------------------------------------------
Fidelity between Pure State Simulation and MCWF Simulation Results:	 0.9844459876673165
Fidelity between Pure State Simulation and Hardware Results:	 0.9753101579582508
Fidelity between MCWF Simulation and Hardware Results:	 0.994100911072567
--------------------------------------------------------------------------------